In [54]:
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [55]:
import os
os.chdir(r'C:\Kaggle_Competition\Playground\S6E6-STELLAR-CLASS')
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import gc
import json
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import log_loss, balanced_accuracy_score
from sklearn.inspection import permutation_importance
import mlflow
from tqdm.notebook import tqdm
import itertools
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import TargetEncoder, OneHotEncoder, StandardScaler
# Models
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import HistGradientBoostingClassifier, ExtraTreesClassifier
from lightgbm import LGBMClassifier
from lightgbm import early_stopping, log_evaluation
from xgboost import XGBClassifier
from xgboost.callback import EarlyStopping
from catboost import CatBoostClassifier
from pytabkit import RealMLP_TD_Classifier, TabM_D_Classifier, Resnet_RTDL_D_Classifier, TabM_HPO_Classifier, RealMLP_HPO_Classifier, FTT_D_Classifier
from sklearn.manifold import TSNE

import tabpfn_client
from tabpfn_client import TabPFNClassifier

from autogluon.tabular import TabularPredictor


import config
import warnings
warnings.filterwarnings('ignore')

In [56]:
train = pd.read_csv('data/raw/train.csv')
test = pd.read_csv('data/raw/test.csv')
orig = pd.read_csv('data/orig/star_classification.csv')

train_ids = train[config.ID_COL].values
test_ids = test[config.ID_COL].values

print(f'Shape of train data: {train.shape}')
print(f'Shape of test data: {test.shape}')
print(f'Shape of orig data: {orig.shape}')

Shape of train data: (577347, 12)
Shape of test data: (247435, 11)
Shape of orig data: (100000, 18)


In [57]:
# Quantitative features
num_cols = ['alpha', 'delta', 'u', 'g', 'r', 'i', 'z', 'redshift']

# Qualitative features
cat_cols = ['spectral_type', 'galaxy_population']

# Target column
target = 'class'

In [58]:
# Data
X = train.drop(['id', config.TARGET_COL], axis=1)
y = train[config.TARGET_COL]
X_test = test.drop('id', axis=1)

### Logistic Regression

In [59]:
# Predictions
oof_predictions = np.zeros((len(train), config.N_CLASSES), dtype=np.float32)
test_predictions = np.zeros((len(X_test), config.N_CLASSES), dtype=np.float32)

# Experiment Setup
EXPERIMENT_NAME = "lr"
VERSION = "v1"
RUN_NAME = f"{EXPERIMENT_NAME}_{VERSION}_seed{config.SEED}"

mlflow.set_experiment(EXPERIMENT_NAME)

# CV
skf = StratifiedKFold(
    n_splits=config.N_FOLDS,
    shuffle=True,
    random_state=config.SEED
)

fold_loglosses = []
fold_baccs = []

feature_importances = []
test_weight = 1 / config.N_FOLDS

with mlflow.start_run(run_name=RUN_NAME):
    for fold, (train_indices, valid_indices) in tqdm(
        enumerate(skf.split(X, y), start=1),
        desc="Model Training",
        total=config.N_FOLDS
    ):

        # Train and validation data
        X_train = X.iloc[train_indices]
        X_valid = X.iloc[valid_indices]

        y_train = y.iloc[train_indices]
        y_valid = y.iloc[valid_indices]

        # Preprocessor
        preprocess = ColumnTransformer(
            transformers=[
                ("cat", OneHotEncoder(handle_unknown="ignore"), cat_cols),
                ("num", StandardScaler(), num_cols)
            ], remainder='passthrough'
        )

        model = Pipeline(
            steps=[
                ("preprocess", preprocess),
                ("classifier", LogisticRegression(**config.LR_PARAMS))
            ]
        )

        # Training model
        model.fit(X_train, y_train)

        perm = permutation_importance(
            estimator=model,
            X=X_valid,
            y=y_valid,
            scoring="balanced_accuracy",
            n_repeats=5,
            random_state=config.SEED,
            n_jobs=-1
        )

        feature_importances.append(perm.importances_mean)

        # Fold level predictions
        fold_oof_predictions = model.predict_proba(X_valid)
        fold_test_predictions = model.predict_proba(X_test)

        # Updating global predictions
        oof_predictions[valid_indices] = fold_oof_predictions
        test_predictions += (fold_test_predictions * test_weight)

        # Inference and scoring
        fold_logloss = log_loss(y_valid, fold_oof_predictions)
        fold_preds = model.predict(X_valid)

        fold_bacc = balanced_accuracy_score(y_valid, fold_preds)

        fold_loglosses.append(fold_logloss)
        fold_baccs.append(fold_bacc)

        print(f"Fold: {fold} | LogLoss: {fold_logloss:.5f} | BACC: {fold_bacc:.5f}")

    # Calculating important metrices
    oof_logloss = log_loss(y, oof_predictions)
    oof_preds = model.classes_[np.argmax(oof_predictions, axis=1)]
    oof_bacc = balanced_accuracy_score(y, oof_preds)
    std_oof_logloss = np.std(fold_loglosses)
    std_oof_bacc = np.std(fold_baccs)

    # Mlflow metrics logging
    mlflow.log_metric("oof_logloss", oof_logloss)
    mlflow.log_metric("oof_bacc", oof_bacc)
    mlflow.log_metric("std_oof_logloss", std_oof_logloss)
    mlflow.log_metric("std_oof_bacc", std_oof_bacc)

    # Mlflow params logging
    mlflow.log_params(config.LR_PARAMS)

    # Mlflow input feature names logging
    mlflow.log_dict({"feature_names": list(X_train.columns)}, "feature_names.json")

    # Feature importance
    feature_importances = pd.DataFrame(feature_importances, columns=X.columns)
    mean_importance = (feature_importances.mean(axis=0).sort_values(ascending=False).head(50))
    std_importance = (feature_importances.std(axis=0).loc[mean_importance.index])

    importance_df = pd.DataFrame({
        "feature":
            mean_importance.index,
        "importance":
            mean_importance.values,
        "std":
            std_importance.values
    })

    fig, ax = plt.subplots(figsize=(10, 6))

    ax.barh(
        mean_importance.index[::-1],
        mean_importance.values[::-1],
        xerr=std_importance.values[::-1]
    )

    ax.set_title("Cross-Validated Permutation Feature Importance")
    ax.set_xlabel("Importance (Balanced Accuracy Drop)")
    plt.tight_layout()

    # Mlflow logging feature imp plot and table
    mlflow.log_figure(fig, "permutation_importance.png")
    mlflow.log_table(importance_df, "permutation_importance.json")
    plt.close(fig)

# Saving oof predictions
oof_proba_df = pd.DataFrame(oof_predictions, columns=model.classes_)
oof_proba_df.insert(0, "id", train_ids)

oof_path = Path(config.OOF_PROBA_DIR, f"{RUN_NAME}_oof_proba.csv")
oof_proba_df.to_csv(oof_path, index=False)

# Saving test predictions
test_proba_df = pd.DataFrame(test_predictions, columns=model.classes_)
test_proba_df.insert(0, "id", test_ids)

test_path = Path(config.TEST_PROBA_DIR, f"{RUN_NAME}_test_proba.csv")
test_proba_df.to_csv(test_path, index=False)

# Sanity check
oof_sanity = pd.read_csv(oof_path)
preds = model.classes_[np.argmax(oof_sanity[model.classes_].values, axis=1)]
print(f"Sanity check completed! BACC score: {balanced_accuracy_score(y, preds):.5f}")

2026/06/01 23:45:13 INFO mlflow.tracking.fluent: Experiment with name 'lr' does not exist. Creating a new experiment.


Model Training:   0%|          | 0/5 [00:00<?, ?it/s]

Fold: 1 | LogLoss: 0.26381 | BACC: 0.91251
Fold: 2 | LogLoss: 0.26685 | BACC: 0.91193
Fold: 3 | LogLoss: 0.26874 | BACC: 0.91209
Fold: 4 | LogLoss: 0.26792 | BACC: 0.91209
Fold: 5 | LogLoss: 0.26947 | BACC: 0.91129
Sanity check completed! BACC score: 0.91198


### HistGBM

In [ ]:
# Predictions
oof_predictions = np.zeros((len(train), config.N_CLASSES), dtype=np.float32)
test_predictions = np.zeros((len(X_test), config.N_CLASSES), dtype=np.float32)

# Experiment Setup
EXPERIMENT_NAME = "histgbm"
VERSION = "v1"
RUN_NAME = f"{EXPERIMENT_NAME}_{VERSION}_seed{config.SEED}"

mlflow.set_experiment(EXPERIMENT_NAME)

# CV
skf = StratifiedKFold(
    n_splits=config.N_FOLDS,
    shuffle=True,
    random_state=config.SEED
)

fold_loglosses = []
fold_baccs = []

feature_importances = []
test_weight = 1 / config.N_FOLDS

with mlflow.start_run(run_name=RUN_NAME):
    for fold, (train_indices, valid_indices) in tqdm(
        enumerate(skf.split(X, y), start=1),
        desc="Model Training",
        total=config.N_FOLDS
    ):

        # Train and validation data
        X_train = X.iloc[train_indices]
        X_valid = X.iloc[valid_indices]

        y_train = y.iloc[train_indices]
        y_valid = y.iloc[valid_indices]

        # Preprocessor
        preprocess = ColumnTransformer(
            transformers=[
                ("cat", OneHotEncoder(handle_unknown="ignore"), cat_cols)
            ], remainder='passthrough'
        )

        model = Pipeline(
            steps=[
                ("preprocess", preprocess),
                ("classifier", HistGradientBoostingClassifier(**config.HISTGBM_PARAMS))
            ]
        )

        # Training model
        model.fit(X_train, y_train)

        perm = permutation_importance(
            estimator=model,
            X=X_valid,
            y=y_valid,
            scoring="balanced_accuracy",
            n_repeats=5,
            random_state=config.SEED,
            n_jobs=-1
        )

        feature_importances.append(perm.importances_mean)

        # Fold level predictions
        fold_oof_predictions = model.predict_proba(X_valid)
        fold_test_predictions = model.predict_proba(X_test)

        # Updating global predictions
        oof_predictions[valid_indices] = fold_oof_predictions
        test_predictions += (fold_test_predictions * test_weight)

        # Inference and scoring
        fold_logloss = log_loss(y_valid, fold_oof_predictions)
        fold_preds = model.predict(X_valid)

        fold_bacc = balanced_accuracy_score(y_valid, fold_preds)

        fold_loglosses.append(fold_logloss)
        fold_baccs.append(fold_bacc)

        print(f"Fold: {fold} | LogLoss: {fold_logloss:.5f} | BACC: {fold_bacc:.5f}")

    # Calculating important metrices
    oof_logloss = log_loss(y, oof_predictions)
    oof_preds = model.classes_[np.argmax(oof_predictions, axis=1)]
    oof_bacc = balanced_accuracy_score(y, oof_preds)
    std_oof_logloss = np.std(fold_loglosses)
    std_oof_bacc = np.std(fold_baccs)

    # Mlflow metrics logging
    mlflow.log_metric("oof_logloss", oof_logloss)
    mlflow.log_metric("oof_bacc", oof_bacc)
    mlflow.log_metric("std_oof_logloss", std_oof_logloss)
    mlflow.log_metric("std_oof_bacc", std_oof_bacc)

    # Mlflow params logging
    mlflow.log_params(config.HISTGBM_PARAMS)

    # Mlflow input feature names logging
    mlflow.log_dict({"feature_names": list(X_train.columns)}, "feature_names.json")

    # Feature importance
    feature_importances = pd.DataFrame(feature_importances, columns=X.columns)
    mean_importance = (feature_importances.mean(axis=0).sort_values(ascending=False).head(50))
    std_importance = (feature_importances.std(axis=0).loc[mean_importance.index])

    importance_df = pd.DataFrame({
        "feature":
            mean_importance.index,
        "importance":
            mean_importance.values,
        "std":
            std_importance.values
    })

    fig, ax = plt.subplots(figsize=(10, 6))

    ax.barh(
        mean_importance.index[::-1],
        mean_importance.values[::-1],
        xerr=std_importance.values[::-1]
    )

    ax.set_title("Cross-Validated Permutation Feature Importance")
    ax.set_xlabel("Importance (Balanced Accuracy Drop)")
    plt.tight_layout()

    # Mlflow logging feature imp plot and table
    mlflow.log_figure(fig, "permutation_importance.png")
    mlflow.log_table(importance_df, "permutation_importance.json")
    plt.close(fig)

# Saving oof predictions
oof_proba_df = pd.DataFrame(oof_predictions, columns=model.classes_)
oof_proba_df.insert(0, "id", train_ids)

oof_path = Path(config.OOF_PROBA_DIR, f"{RUN_NAME}_oof_proba.csv")
oof_proba_df.to_csv(oof_path, index=False)

# Saving test predictions
test_proba_df = pd.DataFrame(test_predictions, columns=model.classes_)
test_proba_df.insert(0, "id", test_ids)

test_path = Path(config.TEST_PROBA_DIR, f"{RUN_NAME}_test_proba.csv")
test_proba_df.to_csv(test_path, index=False)

# Sanity check
oof_sanity = pd.read_csv(oof_path)
preds = model.classes_[np.argmax(oof_sanity[model.classes_].values, axis=1)]
print(f"Sanity check completed! BACC score: {balanced_accuracy_score(y, preds):.5f}")